Find the files

In [2]:
from pathlib import Path

DATA_DIR_PARQ = Path(r"C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7")
DATA_DIR_CSV = Path(r"C:\Users\adiz\Downloads\archive(1)")

parquet_files = sorted(DATA_DIR_PARQ.glob("*.parquet"))
csv_files = sorted(DATA_DIR_CSV.glob("*.csv"))

print("Parquet files:")
for f in parquet_files:
    print(f)

print("\nCSV files:")
for f in csv_files:
    print(f)


Parquet files:
C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7\transactions_00.parquet
C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7\transactions_01.parquet
C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7\transactions_02.parquet
C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7\transactions_03.parquet
C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7\transactions_04.parquet
C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7\transactions_05.parquet
C:\Users\adiz\Downloads\archive(1)\credit_card_transactions_parquet_7\transactions_06.parquet

CSV files:
C:\Users\adiz\Downloads\archive(1)\credit_card_transactions-ibm_v2.csv
C:\Users\adiz\Downloads\archive(1)\sd254_cards.csv
C:\Users\adiz\Downloads\archive(1)\sd254_users.csv
C:\Users\adiz\Downloads\archive(1)\User0_credit_card_transactions.csv


Inspect the Parquet files without loading everything

In [3]:
import pandas as pd

for file in parquet_files:
    df = pd.read_parquet(file)

    print(
        f"{file.name:30s} "
        f"rows={len(df):,} "
        f"columns={len(df.columns)}"
    )


transactions_00.parquet        rows=3,500,000 columns=15
transactions_01.parquet        rows=3,500,000 columns=15
transactions_02.parquet        rows=3,500,000 columns=15
transactions_03.parquet        rows=3,500,000 columns=15
transactions_04.parquet        rows=3,500,000 columns=15
transactions_05.parquet        rows=3,500,000 columns=15
transactions_06.parquet        rows=3,386,900 columns=15


Check that all 7 files have the same columns

In [7]:
schemas = {}

for file in parquet_files:
    df = pd.read_parquet(file)

    schemas[file.name] = {
        column: str(dtype)
        for column, dtype in df.dtypes.items()
    }

for file, schema in schemas.items():
    print(f"\n{file}")
    print("-" * 60)

    for column, dtype in schema.items():
        print(f"{column:30s} {dtype}")





transactions_00.parquet
------------------------------------------------------------
User                           int64
Card                           int64
Year                           int64
Month                          int64
Day                            int64
Time                           str
Amount                         str
Use Chip                       str
Merchant Name                  int64
Merchant City                  str
Merchant State                 str
Zip                            float64
MCC                            int64
Errors?                        str
Is Fraud?                      str

transactions_01.parquet
------------------------------------------------------------
User                           int64
Card                           int64
Year                           int64
Month                          int64
Day                            int64
Time                           str
Amount                         str
Use Chip                      

In [9]:
reference_file = parquet_files[0]
reference_df = pd.read_parquet(reference_file)

reference_columns = set(reference_df.columns)

for file in parquet_files[1:]:
    df = pd.read_parquet(file)

    columns = set(df.columns)

    missing_columns = reference_columns - columns
    extra_columns = columns - reference_columns

    if missing_columns or extra_columns:
        print(f"\n{file.name}")

        if missing_columns:
            print("Missing columns:", missing_columns)

        if extra_columns:
            print("Extra columns:", extra_columns)


Missing-value analysis across all 7 files

In [ ]:
'''
Emplainable missing values only in two situations: the purchase is made A) online or B) outside the U.S:

In case A)
Both Zip + Merchant State are empty

In case B)
"Merchant State" has a country name + Zip is empty (as zip is only supported for U.S.)
'''

for file in parquet_files:
    df = pd.read_parquet(file)

    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100,
    })

    missing = missing[missing["missing_count"] > 0]
    missing = missing.sort_values("missing_count", ascending=False)

    print(f"\n{'=' * 80}")
    print(file.name)
    print(f"{'=' * 80}")

    display(missing)



transactions_00.parquet


,missing_count,missing_pct
Errors?,3442757,98.364486
Zip,424244,12.121257
Merchant State,401229,11.463686



transactions_01.parquet


,missing_count,missing_pct
Errors?,3442338,98.352514
Zip,404549,11.558543
Merchant State,381739,10.906829



transactions_02.parquet


...

,missing_count,missing_pct
Errors?,3443868,98.396229
Zip,380506,10.871600
Merchant State,356618,10.189086



transactions_05.parquet


,missing_count,missing_pct
Errors?,3445667,98.447629
Zip,429281,12.265171
Merchant State,408203,11.662943



transactions_06.parquet


,missing_count,missing_pct
Errors?,3333415,98.420827
Zip,417172,12.317222
Merchant State,396979,11.721013


Check the actual transaction columns

In [11]:
transactions_sample = pd.read_parquet(parquet_files[0])

print(transactions_sample.columns.tolist())


['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'Is Fraud?']


In [12]:
display(
    pd.DataFrame({
        "column": transactions_sample.columns,
        "dtype": transactions_sample.dtypes.astype(str),
        "unique": transactions_sample.nunique().values,
        "missing": transactions_sample.isna().sum().values,
        "missing_pct": (
            transactions_sample.isna().mean().values * 100
        )
    })
)


,column,dtype,unique,missing,missing_pct
User,User,int64,276,0,0.000000
Card,Card,int64,8,0,0.000000
Year,Year,int64,28,0,0.000000
Month,Month,int64,12,0,0.000000
Day,Day,int64,31,0,0.000000
Time,Time,str,1440,0,0.000000
Amount,Amount,str,51093,0,0.000000
Use Chip,Use Chip,str,3,0,0.000000
Merchant Name,Merchant Name,int64,27739,0,0.000000
Merchant City,Merchant City,str,8708,0,0.000000


Read users and cards separately

In [14]:
users = pd.read_csv(DATA_DIR_CSV / "sd254_users.csv")
cards = pd.read_csv(DATA_DIR_CSV / "sd254_cards.csv")

print("Users:", users.shape)
print("Cards:", cards.shape)


Users: (2000, 18)
Cards: (6146, 13)


In [15]:
print("Users columns:")
print(users.columns.tolist())

print("\nCards columns:")
print(cards.columns.tolist())


Users columns:
['Person', 'Current Age', 'Retirement Age', 'Birth Year', 'Birth Month', 'Gender', 'Address', 'Apartment', 'City', 'State', 'Zipcode', 'Latitude', 'Longitude', 'Per Capita Income - Zipcode', 'Yearly Income - Person', 'Total Debt', 'FICO Score', 'Num Credit Cards']

Cards columns:
['User', 'CARD INDEX', 'Card Brand', 'Card Type', 'Card Number', 'Expires', 'CVV', 'Has Chip', 'Cards Issued', 'Credit Limit', 'Acct Open Date', 'Year PIN last Changed', 'Card on Dark Web']


In [16]:
display(users.head())
display(cards.head())


,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5
1,Sasha Sadr,53,68,1966,12,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5
2,Saanvi Lee,81,67,1938,11,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5
3,Everlee Clark,63,63,1957,1,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,$163145,$249925,$202328,722,4
4,Kyle Peterson,43,70,1976,9,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,$53797,$109687,$183855,675,1


,User,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,0,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,0,1,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,0,2,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,0,3,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,0,4,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [17]:
print("USERS")
display(
    users.isna()
    .sum()
    .to_frame("missing")
    .assign(
        missing_pct=lambda x: x["missing"] / len(users) * 100
    )
    .sort_values("missing", ascending=False)
)

print("CARDS")
display(
    cards.isna()
    .sum()
    .to_frame("missing")
    .assign(
        missing_pct=lambda x: x["missing"] / len(cards) * 100
    )
    .sort_values("missing", ascending=False)
)


USERS


,missing,missing_pct
Apartment,1472,73.6
Person,0,0.0
Current Age,0,0.0
Retirement Age,0,0.0
Birth Month,0,0.0
Birth Year,0,0.0
Gender,0,0.0
Address,0,0.0
City,0,0.0
State,0,0.0


CARDS


,missing,missing_pct
User,0,0.0
CARD INDEX,0,0.0
Card Brand,0,0.0
Card Type,0,0.0
Card Number,0,0.0
Expires,0,0.0
CVV,0,0.0
Has Chip,0,0.0
Cards Issued,0,0.0
Credit Limit,0,0.0


Referential-integrity check to determine whether every transaction's card/user identifier actually exists in the corresponding CSVs

In [21]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")

# Load the relatively small dimension tables
users = pd.read_csv(DATA_DIR_CSV / "sd254_users.csv")
cards = pd.read_csv(DATA_DIR_CSV / "sd254_cards.csv")

print("Users:", users.shape)
print("Cards:", cards.shape)

print("\nUsers columns:")
print(users.columns.tolist())

print("\nCards columns:")
print(cards.columns.tolist())

#Check facts table
print("\nTransactions columns:")
print(transactions_sample.columns.tolist())


Users: (2000, 18)
Cards: (6146, 13)

Users columns:
['Person', 'Current Age', 'Retirement Age', 'Birth Year', 'Birth Month', 'Gender', 'Address', 'Apartment', 'City', 'State', 'Zipcode', 'Latitude', 'Longitude', 'Per Capita Income - Zipcode', 'Yearly Income - Person', 'Total Debt', 'FICO Score', 'Num Credit Cards']

Cards columns:
['User', 'CARD INDEX', 'Card Brand', 'Card Type', 'Card Number', 'Expires', 'CVV', 'Has Chip', 'Cards Issued', 'Credit Limit', 'Acct Open Date', 'Year PIN last Changed', 'Card on Dark Web']

Transactions columns:
['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'Is Fraud?']


In [22]:
# Check referential integrity:
# cards:        User + CARD INDEX
# transactions: User + Card

# Create the set of valid (User, CARD INDEX) combinations from cards.csv
valid_card_keys = set(
    zip(
        cards["User"],
        cards["CARD INDEX"]
    )
)

print(f"Unique valid User + CARD INDEX combinations: {len(valid_card_keys):,}")


# Check every transaction parquet file
integrity_results = []

for file in parquet_files:

    transactions = pd.read_parquet(file)

    # Create (User, Card) pairs for transactions
    transaction_keys = list(
        zip(
            transactions["User"],
            transactions["Card"]
        )
    )

    # Check whether each transaction's User + Card exists in cards
    exists_in_cards = [
        key in valid_card_keys
        for key in transaction_keys
    ]

    exists_in_cards = pd.Series(
        exists_in_cards,
        index=transactions.index
    )

    invalid_count = (~exists_in_cards).sum()

    integrity_results.append({
        "file": file.name,
        "transactions": len(transactions),
        "valid": exists_in_cards.sum(),
        "invalid": invalid_count,
        "invalid_pct": invalid_count / len(transactions) * 100
    })


# Display results
integrity_report = pd.DataFrame(integrity_results)

display(integrity_report)


Unique valid User + CARD INDEX combinations: 6,146


,file,transactions,valid,invalid,invalid_pct
0,transactions_00.parquet,3500000,3500000,0,0.0
1,transactions_01.parquet,3500000,3500000,0,0.0
2,transactions_02.parquet,3500000,3500000,0,0.0
3,transactions_03.parquet,3500000,3500000,0,0.0
4,transactions_04.parquet,3500000,3500000,0,0.0
5,transactions_05.parquet,3500000,3500000,0,0.0
6,transactions_06.parquet,3386900,3386900,0,0.0


This relationship can be established:

transactions.User
        +
transactions.Card
        │
        ▼
cards.User
        +
cards.CARD INDEX


check that the User and Card combination is unique in cards.csv

In [23]:
# Check whether (User, CARD INDEX) uniquely identifies a card

duplicate_card_keys = (
    cards
    .groupby(["User", "CARD INDEX"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
)

print(f"Duplicate (User, CARD INDEX) combinations: {len(duplicate_card_keys):,}")

display(duplicate_card_keys.head(20))


Duplicate (User, CARD INDEX) combinations: 0


,User,CARD INDEX,count


Then we can say that (User, CARD INDEX) is a proper composite key for the cards table.

Vice versa: Card values are unique within each user?

In [25]:
print(
    cards.duplicated(
        subset=["User", "CARD INDEX"]
    ).sum()
)


0


users
  │
  │ User
  ▼
cards
  │
  │ User + CARD INDEX
  ▼
transactions
